# Qwen3-ASR Speech Recognition with OpenVINO™

The Qwen3-ASR family includes Qwen3-ASR-1.7B and Qwen3-ASR-0.6B, which support language identification and ASR for 52 languages and dialects. Both leverage large-scale speech training data and the strong audio understanding capability of their foundation model, Qwen3-Omni. Experiments show that the 1.7B version achieves state-of-the-art performance among open-source ASR models and is competitive with the strongest proprietary commercial APIs. Here are the main features:

* **All-in-one**: Qwen3-ASR-1.7B and Qwen3-ASR-0.6B support language identification and speech recognition for 30 languages and 22 Chinese dialects, so as to English accents from multiple countries and regions.

* **Excellent and Fast**: The Qwen3-ASR family ASR models maintains high-quality and robust recognition under complex acoustic environments and challenging text patterns. Qwen3-ASR-1.7B achieves strong performance on both open-sourced and internal benchmarks. While the 0.6B version achieves accuracy-efficient trade-off, it reaches 2000 times throughput at a concurrency of 128. They both achieve streaming / offline unified inference with single model and support transcribe long audio.

* **Novel and strong forced alignment Solution**: We introduce Qwen3-ForcedAligner-0.6B, which supports timestamp prediction for arbitrary units within up to 5 minutes of speech in 11 languages. Evaluations show its timestamp accuracy surpasses E2E based forced-alignment models.

* **Comprehensive inference toolkit**: In addition to open-sourcing the architectures and weights of the Qwen3-ASR series, we also release a powerful, full-featured inference framework that supports vLLM-based batch inference, asynchronous serving, streaming inference, timestamp prediction, and more.

<p align="center">
    <img src="https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/overview.jpg" width="100%"/>
<p>

More details can be found in the original [repository](https://github.com/QwenLM/Qwen3-ASR) and [model card](https://huggingface.co/Qwen/Qwen3-ASR-0.6B)

In this tutorial, we will:
1. Install required dependencies
2. Download and convert Qwen3-ASR model to OpenVINO format
3. Create an inference pipeline using OpenVINO
4. Build an interactive Gradio demo for speech recognition

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download and Prepare Qwen3-ASR Model](#Download-and-Prepare-Qwen3-ASR-Model)
- [Convert Model to OpenVINO Format](#Convert-Model-to-OpenVINO-Format)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Speech Recognition](#Run-Speech-Recognition)
- [Interactive Demo](#Interactive-Demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-asr/qwen3-asr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
# Fetch notebook_utils module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-asr.ipynb")

### Install Dependencies
[back to top ⬆️](#Table-of-contents:)

Install required packages for Qwen3-ASR model conversion and inference:

In [2]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform

# Uninstall conflicting versions
%pip uninstall -y -q torch torchaudio transformers qwen-asr

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.8.0",
    "nncf",
    "torchaudio==2.8.0",
    "openvino>=2025.4.0",
    "gradio>=4.0",
    "huggingface_hub",
    "scipy",
    "qwen-asr",
)

# Clone Qwen3-ASR repository for model code
repo_dir = Path("Qwen3-ASR")
revision = "c17a131fe028b2e428b6e80a33d30bb4fa57b8df"
clone_repo("https://github.com/QwenLM/Qwen3-ASR.git", revision)

pip_install(
    "-q -e",
    str(repo_dir),
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

pip_install(
    "-q",
    "evaluate",
    "jiwer",
)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: switching to 'c17a131fe028b2e428b6e80a33d30bb4fa57b8df'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at c17a131 fix streaming unicode

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Download and Prepare Qwen3-ASR Model
[back to top ⬆️](#Table-of-contents:)

Select the Qwen3-ASR model variant to use. The 0.6B model is recommended for faster inference while maintaining good accuracy:

In [ ]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen3-ASR-0.6B",
    "Qwen/Qwen3-ASR-1.7B",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_selector

: 

### Model Architecture

The Qwen3-ASR pipeline consists of several key components:

* **Audio Frontend**: Extracts mel-spectrogram features from raw audio waveform (16kHz sample rate, 80 mel bins)
* **Audio Encoder (Audio Tower)**: Conv2D layers followed by Transformer encoder blocks that process audio features into embeddings
* **Text Embeddings**: Converts text tokens into embeddings for the language model
* **Language Model (LLM)**: A Qwen3-based decoder that takes merged audio and text embeddings and generates transcription output

The model uses a multimodal approach where audio embeddings replace special audio tokens in the text sequence before being processed by the LLM.

## Convert Model to OpenVINO Format
[back to top ⬆️](#Table-of-contents:)

Now we'll convert the Qwen3-ASR model to OpenVINO Intermediate Representation (IR) format. The conversion process exports the following components:

1. **Audio Conv Model** (`openvino_thinker_audio_model.xml`): The Conv2D frontend for audio feature extraction
2. **Audio Encoder Model** (`openvino_thinker_audio_encoder_model.xml`): The Transformer encoder layers
3. **Embedding Model** (`openvino_thinker_embedding_model.xml`): Text token embedding layer
4. **Language Model** (`openvino_thinker_language_model.xml`): The main LLM decoder with KV-cache support

In [3]:
from qwen_3_asr_helper import convert_qwen3_asr_model

from nncf import CompressWeightsMode

model_id = model_selector.value
model_name = model_id.split("/")[-1]
ov_model_dir = Path(f"{model_name}-OV")
# Convert model to OpenVINO format
# This will skip conversion if the model already exists
convert_qwen3_asr_model(
    model_id=model_id,
    output_dir=ov_model_dir,
    quantization_config={"mode": CompressWeightsMode.INT8_SYM},  # Set to {"mode": CompressWeightsMode.INT8_SYM} for INT8 quantization
)

⌛ Qwen/Qwen3-ASR-1.7B conversion started. Be patient, it may takes some time.
⌛ Load Original model


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Original model successfully loaded
⌛ Convert thinker embedding model
✅ Thinker embedding model successfully converted
⌛ Convert thinker audio model (Conv2D part)


...

✅ Thinker audio model (Conv2D part) successfully converted
⌛ Convert thinker audio encoder model (Transformer layers)
✅ Thinker audio encoder model (Transformer layers) successfully converted
⌛ Convert Thinker Language model


/home/user/workspace/openvino_notebooks/notebooks/qwen3-asr/.venv/lib/python3.12/site-packages/transformers/cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:
/home/user/workspace/openvino_notebooks/notebooks/qwen3-asr/Qwen3-ASR/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py:1021: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if position_ids.ndim == 3 and position_ids.shape[0] == 4:
/home/user/workspace/openvino_notebooks/notebooks/qwen3-asr/qwen_3_asr_helper.py:119: TracerW

✅ Thinker language model successfully converted
⌛ Weights compression with int8_sym mode started
INFO:nncf:Statistics of the bitwidth distribution:
┍━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┑
│ Weight compression mode   │ % all parameters (layers)   │ % ratio-defining parameters (layers)   │
┝━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┥
│ int8_sym, per-channel     │ 100% (197 / 197)            │ 100% (197 / 197)                       │
┕━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┙


Output()

✅ Weights compression finished
✅ Thinker model conversion finished. You can find results in Qwen3-ASR-1.7B-OV
✅ Qwen/Qwen3-ASR-1.7B model conversion finished. You can find results in Qwen3-ASR-1.7B-OV


## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference. CPU provides the most compatibility, while GPU and NPU can provide acceleration on supported hardware:

In [4]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU.0', 'GPU.1', 'AUTO'), value='CPU')

### Load OpenVINO Model
[back to top ⬆️](#Table-of-contents:)

Load the converted OpenVINO model using our helper class. The `OVQwen3ASRModel` provides the same API as the original Qwen3ASRModel:

In [5]:
from qwen_3_asr_helper import OVQwen3ASRModel

ov_model = OVQwen3ASRModel.from_pretrained(
    model_dir=str(ov_model_dir),
    device=device.value,
    max_inference_batch_size=-1,  # Batch size limit for inference. -1 means unlimited.
    max_new_tokens=512,  # Maximum number of tokens to generate. Set larger for long audio.
)

# Print supported languages
print(f"Supported languages: {ov_model.get_supported_languages()}")

⌛ Loading audio conv model from Qwen3-ASR-1.7B-OV/thinker/openvino_thinker_audio_model.xml
⌛ Loading audio encoder model from Qwen3-ASR-1.7B-OV/thinker/openvino_thinker_audio_encoder_model.xml
⌛ Loading embedding model from Qwen3-ASR-1.7B-OV/thinker/openvino_thinker_embedding_model.xml
⌛ Loading language model from Qwen3-ASR-1.7B-OV/thinker/openvino_thinker_language_model.xml
✅ Processor loaded successfully
✅ OVQwen3ASRModel initialized successfully
Supported languages: ['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian']


### Run Speech Recognition
[back to top ⬆️](#Table-of-contents:)

Let's test the model with sample audio files. The model supports various audio formats (wav, mp3, flac) and can accept:
- Local file paths
- URLs to audio files
- NumPy arrays with sample rate tuples

In [6]:
reference = "Insertion level, Terminal ileum, Cecum, Ascending colon, Hepatic flexure, Transverse colon, Splenic flexure, Descending colon, Sigmoid Colon, Rectum, Anastomosis, Anus, Premedication, Colon cleansing agent, Preparation time, Morning single dose, Evening single dose, Split dose, Colon cleansing level, Excellent, Good, Fail, Poor finding, A, normal, Negative finding, Negative finding in the observable segment, Poor preparation, B, Hemorrhoids, External Hemorrhoids, Mixed hemorrhoids, Internal hemorrhoids, C, polyp, Hyperplastic polyp, Tubular adenoma, Tubulovillous adenoma, Villous adenoma, Sessile serrated lesion, SSL, Traditional serrated adenoma, Post-treatment residual neoplasm, Inflammatory polyp, Juvenile polyp, Peutz-Jeghers syndrome, Colon polyposis, familiar, Colon polyposis, Early colorectal cancer, Advanced colorectal cancer, Lymphangioma, Lipoma, Carcinoid, Submucosal tumor, Colonmaltoma, Lymphoma, Colitis, Non-specific colitis, Ischemic colitis, Infectious colitis, Amebic colitis, Ulcerative colitis, Radiation colitis, Pseudo-membranous colitis, Drug induced colitis, Cytomegalovirus colitis, CMV colitis, GVHD related colitis, Crohn's disease, Colonic ulcer, Bechet's disease, Proctitis, Hemorrhagic colitis, Colitis aphthosa, Colonic diverticulum, Chronic diverticulosis, Melanosis coloi, Xanthoma, Post partial colectomy, Post left hemicolectomy, Post right hemicolectomy, Situs inversus, Colonic wall cyst, Angiodysplasia, Angiectasia, Lymphoid follicles, Operation scar, Suture granuloma, Petechia, Colonic tuberculosis, Amyloidosis, Mega colon, Rectal varices, Mucosal prolapse, Intussusception, Colon fistula, Post endoscopy treatment scar, Colonic stricture, Rectosigmoid junction RSJ"

In [10]:
import time

sample_audios = [Path(f"sample_{i:02}.wav") for i in range(1,11)]
truths = [
    "The colonoscope was advanced to the terminal ileum through the cecum, and the patient received adequate premedication before the procedure.",
    "A five millimeter hyperplastic polyp was identified in the ascending colon and removed using cold forceps technique.",
    "The colon cleansing level was rated as excellent after the patient followed the split dose preparation protocol.",
    "Examination revealed internal hemorrhoids in the rectum and a small tubular adenoma at the hepatic flexure.",
    "The insertion level reached the cecum, but visualization of the transverse colon was limited due to poor preparation.",
    "Pathology confirmed a sessile serrated lesion (SSL) in the descending colon with no evidence of early colorectal cancer.",
    "Non-specific colitis was observed throughout the sigmoid colon and rectosigmoid junction (RSJ).",
    "The patient's history of post left hemicolectomy was noted, and the anastomosis site at the splenic flexure appeared normal with negative findings.",
    "Multiple colonic diverticula were found in the sigmoid colon, along with melanosis coli throughout the examined segments.",
    "A small lipoma was detected in the transverse colon, and the terminal ileum showed lymphoid follicles which are considered a normal finding."
]
time_takens = []
transcriptions = []

for sample_audio in sample_audios:
    start = time.perf_counter()
    results = ov_model.transcribe(
        audio=str(sample_audio),
        language=None,  # Auto-detect language
        context=reference,
    )
    end = time.perf_counter()
    time_takens.append(end-start)
    transcriptions.append(results[0].text)


import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()
metric = evaluate.load("wer")

normalized_truths = []
normalized_transcriptions = []

for i in range(10):
    normalized_truths.append(normalizer(truths[i]).strip())
    normalized_transcriptions.append(normalizer(transcriptions[i]).strip())

wer = 100 * metric.compute(predictions=transcriptions, references=truths)
normalized_wer = 100 * metric.compute(predictions=normalized_transcriptions, references=normalized_truths)

print(f"WER: {wer}")
print(f"Normalized WER: {normalized_wer}")


import librosa

durations = []
rtfs = []

for i in range(10):
    duration = librosa.get_duration(path=sample_audios[i])
    durations.append(duration)
    rtfs.append(time_takens[i]/duration)
    
print(f"Average RTF: {sum(rtfs)/len(rtfs)}")
print(f"RTF: {rtfs}")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


WER: 1.1173184357541899
Normalized WER: 0.0
Average RTF: 0.3419020248177055
RTF: [0.30361636917052026, 0.2928414015627619, 0.33560636763771373, 0.31155713920658423, 0.3960740269447031, 0.28269962159956386, 0.4423015302126395, 0.32236176019866175, 0.40494422323160595, 0.32701780841230077]


In [11]:
# Download sample audio files
import urllib.request
import time

# sample_audio_en = Path("sample_en.wav")
# if not sample_audio_en.exists():
#     print("Downloading English sample audio...")
#     urllib.request.urlretrieve("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav", sample_audio_en)
#     print(f"Downloaded: {sample_audio_en}")

sample_audio_en = Path("endoscopy_internal.wav")

start = time.perf_counter()
results = ov_model.transcribe(
    audio=str(sample_audio_en),
    language=None,  # Auto-detect language
    # context=reference,
)
end = time.perf_counter()

print(f"Detected Language: {results[0].language}")
print(f"Transcription: {results[0].text}")
print(f"Time taken: {end-start} seconds")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Detected Language: English
Transcription: Insertion level. Sigmoid ileum. Cecum. Ascending colon. Hepatic flexure. Transverse colon. Splenic flexure. Descending colon. Sigmoid colon. Rectum. Anastomosis. Anus. Pre-medication. Colon cleansing agent. Preparation time. Morning single dose. Evening single dose. Split dose. Colon cleansing level. Excellent. Good. Fail. Poor finding. A. Normal. Negative finding. Negative finding in the observable segment. Poor preparation. B. Hemorrhoids. External hemorrhoids. Mixed hemorrhoids. Internal hemorrhoids. C. Polyp. Hyperplastic polyp. Tubular adenoma. Tubulovillous adenoma. Villous adenoma. Sessile serrated lesion. S.S. Traditional serrated adenoma. Post-treatment residual neoplasm. Inflammatory polyp. Juvenile polyp. Peutz-Jeghers syndrome. Colon polypsias. Familial colon polypsias. Early colorectal cancer. Advanced colorectal cancer. Leiomyoma. Lipoma. Cystinoid. Submucosal tumor. Colon nodule. Lymphoma. Colitis. Non-specific colitis. Ischemic 

In [12]:
# pip install evaluate jiwer
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()
metric = evaluate.load("wer")

reference = "Insertion level, Terminal ileum, Cecum, Ascending colon, Hepatic flexure, Transverse colon, Splenic flexure, Descending colon, Sigmoid Colon, Rectum, Anastomosis, Anus, Premedication, Colon cleansing agent, Preparation time, Morning single dose, Evening single dose, Split dose, Colon cleansing level, Excellent, Good, Fail, Poor finding, A, normal, Negative finding, Negative finding in the observable segment, Poor preparation, B, Hemorrhoids, External Hemorrhoids, Mixed hemorrhoids, Internal hemorrhoids, C, polyp, Hyperplastic polyp, Tubular adenoma, Tubulovillous adenoma, Villous adenoma, Sessile serrated lesion, SSL, Traditional serrated adenoma, Post-treatment residual neoplasm, Inflammatory polyp, Juvenile polyp, Peutz-Jeghers syndrome, Colon polyposis, familiar, Colon polyposis, Early colorectal cancer, Advanced colorectal cancer, Lymphangioma, Lipoma, Carcinoid, Submucosal tumor, Colonmaltoma, Lymphoma, Colitis, Non-specific colitis, Ischemic colitis, Infectious colitis, Amebic colitis, Ulcerative colitis, Radiation colitis, Pseudo-membranous colitis, Drug induced colitis, Cytomegalovirus colitis, CMV colitis, GVHD related colitis, Crohn's disease, Colonic ulcer, Bechet's disease, Proctitis, Hemorrhagic colitis, Colitis aphthosa, Colonic diverticulum, Chronic diverticulosis, Melanosis coloi, Xanthoma, Post partial colectomy, Post left hemicolectomy, Post right hemicolectomy, Situs inversus, Colonic wall cyst, Angiodysplasia, Angiectasia, Lymphoid follicles, Operation scar, Suture granuloma, Petechia, Colonic tuberculosis, Amyloidosis, Mega colon, Rectal varices, Mucosal prolapse, Intussusception, Colon fistula, Post endoscopy treatment scar, Colonic stricture, Rectosigmoid junction RSJ"
prediction = results[0].text
normalized_reference = normalizer(reference).strip()
normalized_prediction = normalizer(results[0].text).strip()

wer = 100 * metric.compute(predictions=[prediction], references=[reference])
normalized_wer = 100 * metric.compute(predictions=[normalized_prediction], references=[normalized_reference])

print(f"WER: {wer}")
print(f"Normalized WER: {normalized_wer}")

WER: 60.10362694300518
Normalized WER: 17.587939698492463


In [13]:
import librosa

duration = librosa.get_duration(path=sample_audio_en)
rtf = (end - start) / duration
print(f"\nAudio duration: {duration:.2f}s")
print(f"RTF: {rtf:.4f}")


Audio duration: 218.56s
RTF: 0.1902


## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch an interactive Gradio demo that allows you to:
- Upload audio files (WAV, MP3, FLAC, etc.)
- Record audio from your microphone
- Select target language or use auto-detection
- View transcription results with inference metrics

The demo is based on the official [Qwen3-ASR Hugging Face Space](https://huggingface.co/spaces/Qwen/Qwen3-ASR).

In [ ]:
from gradio_helper import make_demo

# Create Gradio demo
demo = make_demo(ov_model, example_dir=".")

# Launch the demo
# If you are launching remotely, specify server_name and server_port:
#   demo.launch(server_name='your_server_name', server_port=7860)
# If you have any issues launching, try:
#   demo.launch(share=True)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)